<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/site_power_ML_site_to_site.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FULLY DATA-DRIVEN TECHNOLOGY-WISE SITE POWER PREDICTION
# ============================================================
# FEATURES:
# - NO ENGINEERING EQUATIONS
# - NO MANUAL POWER PARAMETERS
# - NO TRAFFIC BANDS
# - 4G TRAFFIC AGGREGATED SITE-WISE
# - 5G TRAFFIC AGGREGATED SITE-WISE
# - 2G / 3G STATIC SITE FEATURES
# - FINAL OUTPUT = 75 x 96 x 7 = 50400 ROWS
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# LOAD FILES FROM GITHUB
# ============================================================

site_db_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/Site%20Database%20from%20Sey.xlsx"
site_power_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/site_power%20from%20Sey.xlsx"
traffic_4g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/4G%20Traffic.xlsx"
traffic_5g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G%20Traffic.xlsx"

# ============================================================
# READ EXCEL FILES
# ============================================================

site_db = pd.read_excel(site_db_url)
site_power = pd.read_excel(site_power_url)
traffic_4g = pd.read_excel(traffic_4g_url)
traffic_5g = pd.read_excel(traffic_5g_url)

# ============================================================
# CHECK DATA
# ============================================================

print(site_db.head(2))

# ============================================================
# RENAME SITE DATABASE COLUMNS
# ============================================================

site_db.columns = [

    '#',
    'Site_ID',
    'Site_Name',
    'RRU_2G',
    'RRU_3G',
    'RRU_4G',
    'AAU_5G',
    'Col_H',
    'Col_I',
    'Boards_4G',
    'Boards_5G',
    'BBU5900',
    'BBU3900',
    'BBU3910'

]

# ============================================================
# AGGREGATE 4G TRAFFIC
# ============================================================

traffic_4g_agg = (

    traffic_4g.groupby(

        [
            'Site_ID',
            'trigger_ID',
            'date',
            'datetime'
        ]

    )['traffic_load_mbps']

    .sum()

    .reset_index()

)

traffic_4g_agg = traffic_4g_agg.rename(

    columns={
        'traffic_load_mbps': 'total_4g_traffic'
    }

)

# ============================================================
# AGGREGATE 5G TRAFFIC
# ============================================================

traffic_5g_agg = (

    traffic_5g.groupby(

        [
            'Site_ID',
            'trigger_ID',
            'date',
            'datetime'
        ]

    )['traffic_load_mbps']

    .sum()

    .reset_index()

)

traffic_5g_agg = traffic_5g_agg.rename(

    columns={
        'traffic_load_mbps': 'total_5g_traffic'
    }

)

# ============================================================
# MERGE 4G TRAFFIC
# ============================================================

final_df = site_power.merge(

    traffic_4g_agg,

    on=[
        'Site_ID',
        'trigger_ID',
        'date',
        'datetime'
    ],

    how='left'

)

# ============================================================
# MERGE 5G TRAFFIC
# ============================================================

final_df = final_df.merge(

    traffic_5g_agg,

    on=[
        'Site_ID',
        'trigger_ID',
        'date',
        'datetime'
    ],

    how='left'

)

# ============================================================
# MERGE SITE DATABASE
# ============================================================

final_df = final_df.merge(

    site_db[

        [
            'Site_ID',
            'RRU_2G',
            'RRU_3G',
            'RRU_4G',
            'AAU_5G',
            'Boards_4G',
            'Boards_5G',
            'BBU5900',
            'BBU3900',
            'BBU3910'
        ]

    ],

    on='Site_ID',

    how='left'

)

# ============================================================
# FILL EMPTY VALUES
# ============================================================

final_df = final_df.fillna(0)

# ============================================================
# FEATURES
# ============================================================

X = final_df[

    [

        'total_4g_traffic',
        'total_5g_traffic',

        'RRU_2G',
        'RRU_3G',

        'RRU_4G',
        'AAU_5G',

        'Boards_4G',
        'Boards_5G',

        'BBU5900',
        'BBU3900',
        'BBU3910'

    ]

]

# ============================================================
# TARGET
# ============================================================

y = final_df['site_power']

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42

)

# ============================================================
# RANDOM FOREST MODEL
# ============================================================

model = RandomForestRegressor(

    n_estimators=100,

    random_state=42,

    n_jobs=-1

)

# ============================================================
# TRAIN MODEL
# ============================================================

model.fit(

    X_train,
    y_train

)

# ============================================================
# PREDICT SITE POWER
# ============================================================

final_df['predicted_site_power'] = model.predict(X)

# ============================================================
# CALCULATE ERROR
# ============================================================

final_df['error'] = (

    final_df['site_power']
    -
    final_df['predicted_site_power']

)

# ============================================================
# CALCULATE ERROR %
# ============================================================

final_df['error_percentage'] = (

    np.abs(

        final_df['error']

    )

    /

    final_df['site_power']

) * 100

# ============================================================
# MODEL PERFORMANCE
# ============================================================

y_pred_test = model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    y_pred_test
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_test
    )
)

mape = np.mean(

    np.abs(

        (
            y_test
            -
            y_pred_test
        )

        /

        y_test

    )

) * 100

r2 = r2_score(
    y_test,
    y_pred_test
)

# ============================================================
# PRINT RESULTS
# ============================================================

print("================================")
print("MODEL PERFORMANCE")
print("================================")

print(f"MAE  : {round(mae,2)}")
print(f"RMSE : {round(rmse,2)}")
print(f"MAPE : {round(mape,2)} %")
print(f"R2   : {round(r2,4)}")

# ============================================================
# FINAL OUTPUT TABLE
# ============================================================

output_df = final_df[

    [

        'Site_ID',
        'trigger_ID',
        'date',
        'datetime',

        'total_4g_traffic',
        'total_5g_traffic',

        'site_power',
        'predicted_site_power',

        'error',
        'error_percentage'

    ]

]

# ============================================================
# EXPORT RESULTS
# ============================================================

output_df.to_excel(

    "Fully_Data_Driven_Site_Wise_Predictions.xlsx",

    index=False

)

# ============================================================
# SHOW RESULTS
# ============================================================

print(output_df.head())

print()
print("Total Rows:", len(output_df))

In [ ]:
# ============================================================
# MODEL PERFORMANCE METRICS
# ============================================================

print("================================")
print("MODEL PERFORMANCE")
print("================================")

print(f"MAE  : {round(mae,2)}")
print(f"RMSE : {round(rmse,2)}")
print(f"MAPE : {round(mape,2)} %")
print(f"R2   : {round(r2,4)}")

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import matplotlib.pyplot as plt

# ============================================================
# ACTUAL VS PREDICTED POWER GRAPH
# ============================================================

plt.figure(figsize=(16,6))

plt.plot(
    output_df['site_power'].values,
    label='Actual Site Power'
)

plt.plot(
    output_df['predicted_site_power'].values,
    label='Predicted Site Power'
)

plt.xlabel('Samples')

plt.ylabel('Power (W)')

plt.title('Actual vs Predicted Site Power')

plt.legend()

plt.grid(True)

plt.show()

# ============================================================
# ACTUAL VS PREDICTED SCATTER PLOT
# ============================================================

plt.figure(figsize=(8,8))

plt.scatter(

    output_df['site_power'],

    output_df['predicted_site_power']

)

plt.xlabel('Actual Site Power')

plt.ylabel('Predicted Site Power')

plt.title('Actual vs Predicted Scatter Plot')

plt.grid(True)

plt.show()

# ============================================================
# ERROR PERCENTAGE HISTOGRAM
# ============================================================

plt.figure(figsize=(10,6))

plt.hist(

    output_df['error_percentage'],

    bins=30

)

plt.xlabel('Error Percentage (%)')

plt.ylabel('Frequency')

plt.title('Prediction Error Distribution')

plt.grid(True)

plt.show()

# ============================================================
# FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({

    'Feature': X.columns,

    'Importance': model.feature_importances_

})

feature_importance = feature_importance.sort_values(

    by='Importance',

    ascending=False

)

plt.figure(figsize=(12,6))

plt.bar(

    feature_importance['Feature'],

    feature_importance['Importance']

)

plt.xlabel('Features')

plt.ylabel('Importance')

plt.title('Feature Importance')

plt.xticks(rotation=45)

plt.grid(True)

plt.show()

# ============================================================
# SITE-WISE ACTUAL VS PREDICTED
# ============================================================

sitewise_df = (

    output_df.groupby('Site_ID')[

        [
            'site_power',
            'predicted_site_power'
        ]

    ]

    .mean()

    .reset_index()

)

plt.figure(figsize=(18,6))

plt.plot(

    sitewise_df['Site_ID'].astype(str),

    sitewise_df['site_power'],

    label='Actual Site Power'

)

plt.plot(

    sitewise_df['Site_ID'].astype(str),

    sitewise_df['predicted_site_power'],

    label='Predicted Site Power'

)

plt.xlabel('Site_ID')

plt.ylabel('Average Power (W)')

plt.title('Site-wise Actual vs Predicted Power')

plt.xticks(rotation=90)

plt.legend()

plt.grid(True)

plt.show()

# ============================================================
# SITE-WISE ABSOLUTE ERROR
# ============================================================

sitewise_df['absolute_error'] = (

    abs(

        sitewise_df['site_power']
        -
        sitewise_df['predicted_site_power']

    )

)

sitewise_df_sorted = sitewise_df.sort_values(

    by='absolute_error',

    ascending=False

)

plt.figure(figsize=(18,6))

plt.bar(

    sitewise_df_sorted['Site_ID'].astype(str),

    sitewise_df_sorted['absolute_error']

)

plt.xlabel('Site_ID')

plt.ylabel('Absolute Error (W)')

plt.title('Site-wise Absolute Error')

plt.xticks(rotation=90)

plt.grid(True)

plt.show()

# ============================================================
# SITE-WISE REAL ERROR
# SORTED BY SITE_ID
# ============================================================

sitewise_df['real_error'] = (

    sitewise_df['site_power']
    -
    sitewise_df['predicted_site_power']

)

# ============================================================
# SORT BY SITE_ID
# ============================================================

sitewise_df_real_sorted = sitewise_df.sort_values(

    by='Site_ID',

    ascending=True

)

# ============================================================
# PLOT
# ============================================================

plt.figure(figsize=(18,6))

plt.bar(

    sitewise_df_real_sorted['Site_ID'].astype(str),

    sitewise_df_real_sorted['real_error']

)

# Zero reference line

plt.axhline(
    y=0,
    linestyle='--'
)

plt.xlabel('Site_ID')

plt.ylabel('Real Error (W)')

plt.title('Site-wise Real Error')

plt.xticks(rotation=90)

plt.grid(True)

plt.show()